# M13 — LoRanPAC equal-budget challenger on Kaggle (train-only)

Use **Settings → Accelerator → GPU** (P100 preferred) and **Internet = On**. Attach `zaphat206/cifar-100`. Kaggle extracts `.zip`, so append `.bin` to the exact M6 and M11 ZIP filenames without recompressing, upload both to one private Dataset, and attach it. Use **Save Version → Save & Run All**. Only `/kaggle/working` is retained.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='002110041b448d64d09e5c80e7a7a004ab3136a8'
WORK_DIR='/kaggle/temp/SOHO-CL'
FEATURE_CACHE_DIR='/kaggle/temp/srq_m13_cifar_features'
OUTPUT_DIR='/kaggle/working/srq_m13_loranpac_output'
EXPORT_PATH='/kaggle/working/srq_generalization_m13_loranpac_train_only.zip'
CONFIG='configs/srq_generalization_m13_loranpac_train_only.json'
RUNNER='tools/srq_generalization_m13.py'
SOURCE_NAMES={'m6':'srq_generalization_m6_width_sweep_train_only.zip','m11':'srq_generalization_m11_adaptive_precision_train_only.zip'}
SOURCE_SHA={'m6':'b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e','m11':'65ce03df4da7041833014628b59aac1167f77bde2d64348b9a8a1e4fe09370a7'}
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Pinned checkout, dependencies, GPU, and portable source hashes.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
assert Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir()
repo=Path(WORK_DIR); repo.parent.mkdir(parents=True,exist_ok=True)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--no-checkout',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Kaggle GPU and restart the session.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m13_loranpac_train_only.json':'0fe77c32c6c242bdae70f3d720a54c624e25c0b1099ad3e263c03d8e178698dc',
 'tools/srq_generalization_m13.py':'a8b090ca4bf8a994e81222c63500fec79e3a1233065f416c886fd86cde24b82e',
 'methods/frontends/loranpac.py':'b468d98981671876bbd223e62ab34ad8430d93f315662b0dabd36ab28ba980f7',
 'methods/frontends/ranpac.py':'6b94532f607d245c0d148d09c1a36b44dbbf964f4ee9cbf05ba6f64826a90e60',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
actual=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(); assert actual==REPO_COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0),'| SOURCE LOCK: PASS',actual)

In [ ]:
# Discover byte-preserved artifacts and raw CIFAR input.
INPUT_ROOT=Path('/kaggle/input'); stage=Path('/kaggle/temp/srq_m13_sources'); stage.mkdir(parents=True,exist_ok=True)
def unique_preserved(name):
    matches=sorted(path for candidate in (name,name+'.bin') for path in INPUT_ROOT.rglob(candidate) if path.is_file())
    assert len(matches)==1,f'Upload exactly one {name}.bin; found {matches}'
    return matches[0]
SOURCE_PATHS={}
for key,name in SOURCE_NAMES.items():
    source=unique_preserved(name); assert sha_raw(source)==SOURCE_SHA[key],(source,sha_raw(source),SOURCE_SHA[key])
    destination=stage/name; shutil.copyfile(source,destination); SOURCE_PATHS[key]=str(destination)
cifar_dirs=sorted({p.parent for p in INPUT_ROOT.rglob('meta') if p.is_file() and (p.parent/'train').is_file() and (p.parent/'test').is_file()})
assert len(cifar_dirs)==1,f'Attach zaphat206/cifar-100 exactly once; found {cifar_dirs}'
CIFAR_ROOT=str(cifar_dirs[0])
candidates=[p for p in INPUT_ROOT.rglob('model.safetensors') if p.is_file() and p.stat().st_size==CHECKPOINT_SIZE and sha_raw(p)==CHECKPOINT_SHA]
if candidates: assert len(candidates)==1; CHECKPOINT_PATH=str(candidates[0])
else:
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE and sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
print('TWO SOURCE ARTIFACTS + CIFAR + CHECKPOINT: PASS')

In [ ]:
# Focused gates, then TRAIN-only feature extraction.
subprocess.run([sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_loranpac_analytic_frontend.py','tests/test_srq_generalization_m13.py','tests/test_ranpac_analytic_frontend.py'],check=True)
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/kaggle/working/unused_m13','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
print('M13 PREFLIGHT + TRAIN CACHE: PASS; test.pt ABSENT')

In [ ]:
# Long resumable run. Completed width/budget JSON units survive later unit failures.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--source-m6-artifact',SOURCE_PATHS['m6'],'--source-m11-artifact',SOURCE_PATHS['m11'],'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
completed=subprocess.run(command); RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m13_results.json'
if not result_path.is_file():
    print('COMPLETED UNITS:',[p.name for p in Path(OUTPUT_DIR).glob('m13_unit_*.json')])
    raise RuntimeError('M13 stopped inside a unit. Re-run this cell in the same session.')
result=json.loads(result_path.read_text()); print('STATUS:',result['status']); print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Paper-ready descriptive table and vector Pareto/rank figure.
import pandas as pd, matplotlib.pyplot as plt
rows=[]
for width,comparison in result['source_comparisons'].items():
    for method,item in comparison.items(): rows.append({'width':int(width),'method':method,'budget':'source','head':'source','rank':None,'AIA':item['validation_aia_percent'],'A_final':item['final_validation_accuracy_percent'],'state_MiB':item['final_total_persistent_bytes']/2**20})
for unit in result['units']:
    for head in ('official_ridge0','matched_m6_ridge'): rows.append({'width':unit['width'],'method':'LoRanPAC-TSVD','budget':unit['budget_target'],'head':head,'rank':unit['rank_contract']['derived_max_rank'],'AIA':unit['validation_aia_percent'][head],'A_final':unit['final_validation_accuracy_percent'][head],'state_MiB':unit['final_total_persistent_bytes']/2**20})
frame=pd.DataFrame(rows); display(frame.sort_values(['width','state_MiB','method','head']))
fig,axes=plt.subplots(1,2,figsize=(11,4.1))
for width in sorted(frame.width.unique()):
    part=frame[frame.width==width]; axes[0].scatter(part.state_MiB,part.AIA,label=f'{width//1000}k')
for unit in result['units']: axes[1].plot([r['task'] for r in unit['records']],[r['effective_rank'] for r in unit['records']],marker='o',label=f"{unit['width']//1000}k/{unit['budget_target']}")
axes[0].set_xlabel('Persistent state (MiB)');axes[0].set_ylabel('Validation AIA (%)');axes[1].set_xlabel('Task');axes[1].set_ylabel('Effective TSVD rank')
for ax in axes: ax.grid(alpha=.25);ax.legend(fontsize=7)
fig.tight_layout();plot=Path(OUTPUT_DIR)/'m13_loranpac_pareto.svg';fig.savefig(plot);plt.show();plt.close(fig)

In [ ]:
# Export a compact auditable artifact; caches and source ZIPs are excluded.
export=Path(EXPORT_PATH); members=[Path(OUTPUT_DIR)/'m13_results.json',Path(OUTPUT_DIR)/'m13_accuracy_state.csv',Path(OUTPUT_DIR)/'m13_loranpac_pareto.svg',Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M13_RUNBOOK.md')];members.extend(sorted(Path(OUTPUT_DIR).glob('m13_unit_*.json')))
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members: archive.write(path,path.name);manifest[path.name]=sha_raw(path)
    archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M13 LoRanPAC train-only','files':manifest},indent=2)+'\n')
assert export.is_file() and zipfile.is_zipfile(export)
print('FINAL ARTIFACT:',export,'SHA-256:',sha_raw(export),'bytes:',export.stat().st_size)
from IPython.display import FileLink,display
display(FileLink(str(export)))
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M13_LORANPAC_TRAIN_ONLY','Integrity/numerical failure: preserve artifact; do not retry from accuracy.'